# EcoTrack — Sustainability & SaaS Analytics

EcoTrack is a fictional platform that helps businesses track resource consumption, estimated CO₂ emissions and sustainability progress.

The data and emissions factors are synthetic and intended for portfolio learning only.


## Business questions

1. What are the latest estimated emissions?
2. Which sources and industries contribute most?
3. Are customers reducing emissions?
4. What are MRR, ARR and churn?
5. Does platform engagement relate to customer retention?


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
companies=pd.read_csv(ROOT/'data/processed/companies_clean.csv',parse_dates=['Signup_Date','Churn_Date'])
monthly=pd.read_csv(ROOT/'data/processed/monthly_sustainability_clean.csv',parse_dates=['Month'])
companies.head()


## 1. Executive KPIs


In [ ]:
latest_month=monthly['Month'].max()
latest=monthly[monthly['Month']==latest_month]
mrr=latest['MRR'].sum()
arr=mrr*12
co2_t=latest['CO2_tonnes'].sum()
active=latest['Company_ID'].nunique()
churn=companies['Churn_Flag'].mean()
pd.DataFrame({'KPI':['Estimated CO2 (t)','MRR','ARR','Active Companies','Overall Churn'],'Value':[f'{co2_t:,.1f}',f'£{mrr:,.2f}',f'£{arr:,.2f}',f'{active:,}',f'{churn:.2%}']})


## 2. What creates the CO₂ estimate?

The project multiplies activity data by simplified illustrative factors. This demonstrates data modelling, not official carbon accounting.


In [ ]:
source_totals=pd.Series({
'Electricity':latest['Electricity_CO2_kg'].sum()/1000,
'Gas':latest['Gas_CO2_kg'].sum()/1000,
'Fuel':latest['Fuel_CO2_kg'].sum()/1000,
'Travel':latest['Travel_CO2_kg'].sum()/1000,
'Waste':latest['Waste_CO2_kg'].sum()/1000
}).sort_values(ascending=False)
source_totals


## 3. Emissions trend


In [ ]:
trend=monthly.groupby('Month',as_index=False)['CO2_tonnes'].sum()
plt.figure(figsize=(10,5))
plt.plot(trend['Month'],trend['CO2_tonnes'],marker='o')
plt.title('Estimated CO2 Emissions Trend')
plt.ylabel('CO2 tonnes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 4. Industry comparison


In [ ]:
latest.groupby('Industry').agg(
CO2_tonnes=('CO2_tonnes','sum'),
Employees=('Employees','sum')
).assign(CO2_kg_per_employee=lambda x:x['CO2_tonnes']*1000/x['Employees']).sort_values('CO2_tonnes',ascending=False)


Using CO₂ per employee gives a second view because total emissions alone can make large companies or industries look worse simply because they have more employees.


## 5. Reduction performance


In [ ]:
companies[['Company_ID','Industry','Reduction_Target_Pct','Observed_Reduction_Pct']].sort_values('Observed_Reduction_Pct',ascending=False).head(15)


## 6. Product engagement and churn


In [ ]:
x=companies.copy()
x['Login_Band']=pd.cut(x['Avg_Monthly_Logins'],[-1,2,5,8,12,100],labels=['0-2','3-5','6-8','9-12','13+'])
x.groupby('Login_Band',observed=False).agg(Companies=('Company_ID','nunique'),Churn_Rate=('Churn_Flag','mean'),Avg_Reduction=('Observed_Reduction_Pct','mean'))


This is the product-analytics part of EcoTrack. If low-engagement customers churn more, the company could build onboarding campaigns, reminders or customer-success alerts.


## 7. Cohort retention


In [ ]:
retention=pd.read_csv(ROOT/'data/processed/cohort_retention.csv')
retention.head(20)


## Interview summary

> I built EcoTrack, a synthetic sustainability SaaS project. I analysed business resource usage and illustrative CO₂ estimates alongside subscription and product-engagement data. I used Python and SQL to analyse emissions sources, industry intensity, reduction performance, MRR, churn and cohort retention, then built an interactive dashboard to communicate the results.
